# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassan-tech-pro/ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane:** Refresh / Content Opportunity Scoring. **Method:** regularized Logistic Regression; rank pages by its probability of the starter decline proxy. A linear, regularized model is a readable first learned scorer, and its probabilities can be evaluated with Precision@K. Numeric missing values are imputed using training-set medians with missingness indicators; categoricals are imputed from training data and one-hot encoded.

**Important boundary:** `decline_proxy` is the starter's current-window trend bucket (`trend_direction == "down"`), not a future outcome. To reduce direct and overlapping-window leakage, the model uses content metadata and freshness fields only; it excludes `trend_direction`, `trend_pct`, impression/click/session totals, and identifiers. Results describe agreement with this proxy, not future content performance or a causal refresh effect.

In [1]:
from pathlib import Path

import pandas as pd

DATA_RELATIVE_PATH = Path("content_refresh_anonymized.csv")
repo_candidates = [Path.cwd(), *Path.cwd().parents]
data_path = next(
    (root / DATA_RELATIVE_PATH for root in repo_candidates if (root / DATA_RELATIVE_PATH).is_file()),
    None,
)
DATA_URL = (
    "https://raw.githubusercontent.com/Hassan-tech-pro/ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)
if data_path is not None:
    raw_data = pd.read_csv(data_path)
    repo_root = data_path.resolve().parents[2]
else:
    raw_data = pd.read_csv(DATA_URL)
    repo_root = Path.cwd()

required_columns = {
    "content_id",
    "client_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
    "trend_pct",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_type",
    "main_intent",
}
missing_columns = required_columns.difference(raw_data.columns)
if missing_columns:
    raise ValueError(f"Starter CSV is missing required columns: {sorted(missing_columns)}")

numeric_columns = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "trend_pct",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
]
for column in numeric_columns:
    raw_data[column] = pd.to_numeric(raw_data[column], errors="coerce")

pages = (
    raw_data.loc[
        raw_data["impressions_90d"].gt(0)
        & raw_data["content_age_days"].ge(90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)
pages["decline_proxy"] = (
    pages["trend_direction"].fillna("").astype(str).str.lower().eq("down").astype("int8")
)

numeric_features = [
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
]
categorical_features = ["content_type", "main_intent"]
feature_columns = numeric_features + categorical_features
target_column = "decline_proxy"
group_column = "client_id"

forbidden_features = {
    "trend_direction",
    "trend_pct",
    target_column,
    "content_id",
    "client_id",
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
}
assert forbidden_features.isdisjoint(feature_columns)

print(f"Source: {'local starter CSV' if data_path is not None else 'public starter CSV'}")
print(f"Raw rows: {len(raw_data):,}; eligible unique pages: {len(pages):,}")
print(f"Client groups: {pages[group_column].nunique():,}")
print(f"Starter decline-proxy prevalence: {pages[target_column].mean():.1%}")
print("Target is a same-snapshot, trend-rule proxy, not a future outcome.")
print(f"Model features: {', '.join(feature_columns)}")

Source: public starter CSV
Raw rows: 30,000; eligible unique pages: 30,000
Client groups: 32
Starter decline-proxy prevalence: 54.2%
Target is a same-snapshot, trend-rule proxy, not a future outcome.
Model features: content_age_days, days_since_last_update, search_volume, competition, cpc, word_count, char_count, content_type, main_intent


## 2. Split design

Use one fixed 75/25 `GroupShuffleSplit` by `client_id` (seed 42, with a small deterministic seed fallback if a holdout lacks both classes). No client appears in both partitions, and IDs are not model features. This tests transfer to unseen clients rather than memorization of pages from already-seen clients. The rule and Logistic Regression use the exact same held-out page rows and target proxy. This starter snapshot is cross-sectional, so it cannot provide a true future-time validation.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

X = pages[feature_columns].copy()
y = pages[target_column].astype(int)
groups = pages[group_column]
if y.nunique() != 2:
    raise ValueError("The starter proxy must contain both classes for a grouped model comparison.")

split_indices = None
for seed in (42, 17, 11, 7, 23):
    candidate = next(
        GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(
            X, y, groups=groups
        )
    )
    candidate_train, candidate_test = candidate
    if y.iloc[candidate_train].nunique() == 2 and y.iloc[candidate_test].nunique() == 2:
        split_indices = candidate
        split_seed = seed
        break
if split_indices is None:
    raise ValueError("Could not produce a client-held-out split containing both proxy classes.")

train_indices, test_indices = split_indices
X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]
train_clients = set(groups.iloc[train_indices])
test_clients = set(groups.iloc[test_indices])
assert train_clients.isdisjoint(test_clients)

print(f"Split seed: {split_seed}; training pages: {len(train_indices):,}; test pages: {len(test_indices):,}")
print(f"Training clients: {len(train_clients)}; held-out clients: {len(test_clients)}")
print(f"Held-out decline-proxy prevalence: {y_test.mean():.1%}")

Split seed: 42; training pages: 22,885; test pages: 7,115
Training clients: 24; held-out clients: 8
Held-out decline-proxy prevalence: 51.7%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(C=1.0, max_iter=2000, random_state=42)),
    ]
)
model.fit(X_train, y_train)
positive_class_index = list(model.classes_).index(1)
model_scores = model.predict_proba(X_test)[:, positive_class_index]

# Reproduce the Week-4 rule: stale and visible, ranked by trailing-90-day impressions.
BASELINE_MIN_IMPRESSIONS_90D = 300
BASELINE_MIN_DAYS_SINCE_UPDATE = 180
heldout_pages = pages.iloc[test_indices]
baseline_candidates = (
    heldout_pages["impressions_90d"].ge(BASELINE_MIN_IMPRESSIONS_90D)
    & heldout_pages["days_since_last_update"].ge(BASELINE_MIN_DAYS_SINCE_UPDATE)
)
baseline_scores = np.where(
    baseline_candidates,
    heldout_pages["impressions_90d"].fillna(0).to_numpy(dtype=float),
    0.0,
)

def precision_at_k(y_true, scores, k):
    truth = np.asarray(y_true, dtype=int)
    ranking_scores = np.asarray(scores, dtype=float)
    if len(truth) < k:
        raise ValueError(f"Precision@{k} requires at least {k} held-out pages; found {len(truth)}.")
    order = np.argsort(-ranking_scores, kind="mergesort")[:k]
    return float(truth[order].mean())

comparison_rows = []
for method_name, scores in [
    ("Week-4 stale-visible rule", baseline_scores),
    ("Logistic Regression", model_scores),
]:
    comparison_rows.append(
        {
            "method": method_name,
            "Precision@10": precision_at_k(y_test, scores, 10),
            "Precision@50": precision_at_k(y_test, scores, 50),
            "Average precision": average_precision_score(y_test, scores),
            "Held-out proxy rate": float(y_test.mean()),
        }
    )

comparison_table = pd.DataFrame(comparison_rows).set_index("method")
display(comparison_table)
print("Both methods use the same eligible pages, held-out clients, proxy labels, and metrics.")
print("Target meaning: agreement with a contemporaneous trend proxy, not future prediction.")
precision50_delta = (
    comparison_table.loc["Logistic Regression", "Precision@50"]
    - comparison_table.loc["Week-4 stale-visible rule", "Precision@50"]
)
if precision50_delta > 0:
    print(f"Logistic Regression exceeds the rule by {precision50_delta:.3f} at Precision@50.")
elif precision50_delta < 0:
    print(f"The rule exceeds Logistic Regression by {-precision50_delta:.3f} at Precision@50.")
else:
    print("The two methods tie at Precision@50 on this held-out split.")

,Precision@10,Precision@50,Average precision,Held-out proxy rate
method,,,,
Week-4 stale-visible rule,0.8,0.70,0.517041,0.516514
Logistic Regression,0.8,0.64,0.528778,0.516514


Both methods use the same eligible pages, held-out clients, proxy labels, and metrics.
Target meaning: agreement with a contemporaneous trend proxy, not future prediction.
The rule exceeds Logistic Regression by 0.060 at Precision@50.


## 4. Errors and interpretation

Use a 0.5 probability threshold only to describe false positives and false negatives; ranking metrics above use model probabilities. The next cell reports three anonymous error examples where available, summarizes error rates by content type, and uses permutation importance on the held-out clients. Importance is a diagnostic association, not causal evidence. Review whether the strongest metadata signals make sense and whether missingness or client mix could explain them.

In [4]:
import numpy as np
import pandas as pd

from sklearn.inspection import permutation_importance

model_predictions = (model_scores >= 0.5).astype(int)
test_review = pages.iloc[test_indices][
    [
        "content_type",
        "content_age_days",
        "days_since_last_update",
        "search_volume",
        "word_count",
    ]
].reset_index(drop=True)
test_review["proxy_true"] = y_test.reset_index(drop=True)
test_review["model_probability"] = model_scores
test_review["model_prediction_at_0_5"] = model_predictions
test_review["error_type"] = np.select(
    [
        (test_review["model_prediction_at_0_5"] == 1) & (test_review["proxy_true"] == 0),
        (test_review["model_prediction_at_0_5"] == 0) & (test_review["proxy_true"] == 1),
    ],
    ["False positive", "False negative"],
    default="Correct",
)

errors = test_review.loc[test_review["error_type"] != "Correct"].copy()
if errors.empty:
    print("No threshold errors at 0.5 on this holdout; there are no wrong cases to display.")
else:
    error_examples = errors.assign(
        confidence_distance=(errors["model_probability"] - 0.5).abs()
    ).sort_values("confidence_distance", ascending=False).head(3)
    error_examples = error_examples.drop(columns="confidence_distance").copy()
    error_examples["why_this_is_hard"] = np.where(
        error_examples["error_type"].eq("False positive"),
        "Metadata resembled pages with the decline proxy, but the current trend bucket was not down.",
        "The metadata did not distinguish this page from non-declining pages despite the down proxy.",
    )
    error_columns = [
        "error_type",
        "content_type",
        "content_age_days",
        "days_since_last_update",
        "search_volume",
        "word_count",
        "model_probability",
        "why_this_is_hard",
    ]
    print(f"Anonymous high-confidence error examples (up to 3 of {len(errors):,} errors):")
    display(error_examples[error_columns])

error_summary = (
    test_review.groupby("content_type", dropna=False)
    .agg(
        n=("proxy_true", "size"),
        proxy_rate=("proxy_true", "mean"),
        threshold_error_rate=("error_type", lambda values: values.ne("Correct").mean()),
    )
    .reset_index()
    .sort_values("n", ascending=False)
)
print("Held-out errors by content type; small groups need cautious interpretation.")
display(error_summary)

permutation = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)
importance_table = (
    pd.DataFrame(
        {
            "feature": feature_columns,
            "mean_average_precision_drop": permutation.importances_mean,
            "std_average_precision_drop": permutation.importances_std,
        }
    )
    .sort_values("mean_average_precision_drop", ascending=False)
    .reset_index(drop=True)
)
print("Held-out permutation importance (mean drop in average precision; not causal).")
display(importance_table.head(5))
leading_features = importance_table.head(3)["feature"].tolist()
print(
    "Top three candidate signals by held-out permutation importance: "
    + ", ".join(leading_features)
)
print("Treat the ordering as noisy when importance is near zero or its spread is large.")

Anonymous high-confidence error examples (up to 3 of 3,339 errors):


,error_type,content_type,content_age_days,days_since_last_update,search_volume,word_count,model_probability,why_this_is_hard
3336,False negative,keyword article,502,20,NaN,NaN,0.121672,The metadata did not distinguish this page fro...
3302,False negative,keyword article,502,20,NaN,NaN,0.121672,The metadata did not distinguish this page fro...
5739,False negative,keyword article,502,22,NaN,NaN,0.122795,The metadata did not distinguish this page fro...


Held-out errors by content type; small groups need cautious interpretation.


,content_type,n,proxy_rate,threshold_error_rate
1,keyword article,6418,0.510439,0.473824
0,comparison article,697,0.572453,0.427547


Held-out permutation importance (mean drop in average precision; not causal).


,feature,mean_average_precision_drop,std_average_precision_drop
0,content_age_days,0.022798,0.003480
1,word_count,0.003592,0.001786
2,competition,0.000630,0.001168
3,search_volume,0.000318,0.001007
4,cpc,0.000147,0.001016


Top three candidate signals by held-out permutation importance: content_age_days, word_count, competition
Treat the ordering as noisy when importance is near zero or its spread is large.


## Self-check

Leave these unchecked until you run the notebook top to bottom and review the outputs yourself:

- [ ] Method fits the refresh-scoring lane and proxy limitations are explicit
- [ ] Split is client-grouped with no client overlap
- [ ] Week-4 stale-visible rule and Logistic Regression use the same held-out rows and Precision@K metrics
- [ ] Comparison includes the held-out proxy rate
- [ ] Errors and held-out feature importance are reviewed skeptically
- [ ] No identifiers, client names, URLs, private queries, or causal claims are exposed
- [ ] Read the research paper linked in the Week-5 card and add your own methodology notes
- [ ] Notebook runs top to bottom; outputs are inspected and explained by me
- [ ] Notebook remains at `work/notebooks/w05_model.ipynb`; no GitHub changes made here